# ARC-3 RFT round 1 (OFFLINE) — LoRA Qwen3.6-27B on RTX Pro 6000, competition-attached

In [ ]:

import sys, glob, os
# Use the fully-resolved dependency bundle (transformers-main + consistent numpy/scipy/sklearn/safetensors/tokenizers)
deps=glob.glob("/kaggle/input/**/deps", recursive=True)
depdir=deps[0] if deps else "/kaggle/input/arc3-deps-prep/deps"
sys.path.insert(0, depdir); print("deps on path:", depdir)
import transformers, peft, trl; print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)


In [ ]:

import torch, glob, os
from transformers import AutoModelForCausalLM, AutoTokenizer
# locate the FP8 27B snapshot dir (has config.json) in the attached model dataset
cfgs=[os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "27b" in p.lower() or "qwen3" in p.lower()]
MODEL=cfgs[0]; print("model dir:", MODEL)
tok=AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
# upcast FP8 -> bf16 for LoRA training (fits 96GB)
model=AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map={"":0}, trust_remote_code=True)
model.config.use_cache=False
# weights are upcast to BF16 on load -> drop quantization metadata so training is plain BF16 LoRA (no torchao)
for a in ("quantization_config","_pre_quantization_dtype"):
    if hasattr(model.config,a):
        try: setattr(model.config,a,None)
        except Exception: pass
model.is_quantized=False
if hasattr(model,"hf_quantizer"): model.hf_quantizer=None
print("loaded:", model.config.model_type, f"{sum(p.numel() for p in model.parameters())/1e9:.1f}B")


In [ ]:

from datasets import load_dataset
import glob
path=glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
ds=load_dataset("json", data_files=path, split="train")
print("RFT examples:", len(ds))


In [ ]:

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                         target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir="/kaggle/working/rft_adapter",
    num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    max_length=3072, packing=False, bf16=True, gradient_checkpointing=True,
    logging_steps=10, save_strategy="epoch", report_to="none",
    assistant_only_loss=True,   # loss on assistant tokens only (needs a chat template with {% generation %})
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=peft_config, processing_class=tok)
trainer.train()
trainer.save_model("/kaggle/working/rft_adapter")
tok.save_pretrained("/kaggle/working/rft_adapter")
print("=== adapter saved to /kaggle/working/rft_adapter ===")
import os; print(os.listdir("/kaggle/working/rft_adapter"))


In [ ]:

# Kaggle expects an output; the adapter dir is the real product.
import pandas as pd
pd.DataFrame([["1_0","1",True,0]],columns=["row_id","game_id","end_of_game","score"]).to_parquet("/kaggle/working/submission.parquet",index=False)
print("done")
